Создает векторное хранилище из текстовых файлов новостей, визуализирует векторы с помощью t-SNE.

In [1]:
import os
import glob
from pathlib import Path
import shutil
from dotenv import load_dotenv
import gradio as gr
import yaml
from tqdm import tqdm

from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np
import plotly.graph_objects as go
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.embeddings import HuggingFaceEmbeddings

In [2]:
# цена является определяющим фактором для нашей компании, поэтому мы собираемся использовать недорогую модель
# MODEL = "gpt-4o-mini"
db_name = "vector_news_db"

# загрузка переменных окружения из .env файла
load_dotenv(override=True)
# os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['HF_API_TOKEN'] = os.getenv('HF_API_TOKEN', 'your_huggingface_token_here')

In [3]:
# Прочитайте документы с помощью загрузчиков LangChain
# Найдите все текстовые файлы в нашей базе знаний

# Получаем список всех файлов .md в папке news и ее подпапках
news_files = list(Path(r"c:/news/").glob("**/*.md"))

def extract_next_bar_from_md(filepath):
    """Извлекает значение 'next_bar' из метаданных Markdown файла."""
    with open(filepath, encoding='utf-8') as f:
        lines = f.readlines()
    if lines[0].strip() == "---":
        # Найти конец YAML-блока
        for i in range(1, len(lines)):
            if lines[i].strip() == "---":
                yaml_block = "".join(lines[1:i])
                meta = yaml.safe_load(yaml_block)
                return meta.get("next_bar")
    return None

def add_metadata(doc, file_path):
    """Добавляет метаданные к документу."""
    # filename = os.path.basename(file_path)
    # filename = file_path.stem  # Имя файла без расширения
    # doc.metadata["date"] = os.path.splitext(filename)[0]  # 
    doc.metadata["date"] = file_path.stem  # Добавление метаданных даты
    next_bar = extract_next_bar_from_md(file_path)  # Извлечение метаданных 'next_bar'
    doc.metadata["next_bar"] = next_bar  # Добавляем метаданные 'next_bar'
    return doc

text_loader_kwargs = {'encoding': 'utf-8'}
# Если это не сработает, некоторым пользователям Windows может потребоваться раскомментировать следующую строку вместо этого
# text_loader_kwargs={'autodetect_encoding': True}

documents = []

for news_file in news_files:
    try:
        # filename = os.path.basename(news_file)
        loader = TextLoader(news_file, encoding=text_loader_kwargs.get('encoding'))
        docs = loader.load()
        for doc in docs:
            doc = add_metadata(doc, news_file)
            documents.append(doc)
    except Exception as e:
        print(f"Ошибка при обработке файла {news_file}: {e}")

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Общее количество фрагментов: {len(chunks)}")
# Выводим только те метаданные, которые у нас есть
if documents:  # Проверяем, что документы были загружены
    print(f"Документы даты: {set(doc.metadata['date'] for doc in documents)}")
    print(f"Направление следующего бара: {set(doc.metadata['next_bar'] for doc in documents)}")
else:
    print("Не удалось найти документы.")

Общее количество фрагментов: 35
Документы даты: {'2025-06-25', '2025-07-09', '2025-07-15', '2025-07-03', '2025-06-26', '2025-06-30', '2025-07-16', '2025-06-27', '2025-07-04', '2025-07-11', '2025-07-14', '2025-07-17', '2025-07-01', 'current', '2025-07-07', '2025-07-08', '2025-07-02', '2025-07-10'}
Направление следующего бара: {'up', 'down', 'current'}


In [4]:
# Поместите фрагменты данных в хранилище векторов, которое связывает векторное вложение с каждым фрагментом
# Chroma - популярная векторная база данных с открытым исходным кодом, основанная на SQLLite

from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
import shutil
import os
from tqdm import tqdm

# Указываем путь к локальной модели
cache_dir = "C:\\Users\\Alkor\\model_cache"

# Загружаем модель (не обязательно передавать в HuggingFaceEmbeddings напрямую)
model = SentenceTransformer(cache_dir)

# Инициализация встраиваний с использованием пути к модели
embeddings = HuggingFaceEmbeddings(model_name=cache_dir)  # Передаем путь как model_name

# Удалить, если уже существует
db_name = "vectorstore"  # Укажите имя директории для векторного хранилища
if os.path.exists(db_name):
    try:
        shutil.rmtree(db_name)  # Удаляет всю директорию и ее содержимое
        print(f"Удалена папка {db_name}")
    except OSError as e:
        print(f"Ошибка при удалении папки {db_name}: {e}")

# Функция для разбиения на батчи
def batch(iterable, n=100):
    """Генератор для разбиения списка на батчи по n элементов"""
    l = len(iterable)
    for ndx in range(0, l, n):
        yield iterable[ndx:min(ndx + n, l)]

# Создание векторного хранилища
vectorstore = None
for chunk_batch in tqdm(list(batch(chunks, 20))):
    if vectorstore is None:
        vectorstore = Chroma.from_documents(
            documents=chunk_batch,
            embedding=embeddings,
            persist_directory=db_name
        )
    else:
        vectorstore.add_documents(chunk_batch)
print(f"Векторное хранилище с {vectorstore._collection.count()} фрагментами документов")

# Исследование векторов
collection = vectorstore._collection
count = collection.count()
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"Есть {count:,} фрагмента документов с {dimensions:,} векторами (размеры в векторном хранилище)")

Удалена папка vectorstore


100%|██████████| 2/2 [00:00<00:00,  2.20it/s]

Векторное хранилище с 35 фрагментами документов
Есть 35 фрагмента документов с 384 векторами (размеры в векторном хранилище)


In [5]:
# Теперь мы можем визуализировать векторы с помощью t-SNE
# t-SNE - это метод, который позволяет визуализировать высокоразмерные данные
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['next_bar'] for metadata in metadatas]
colors = [['blue', 'red', 'black'][['up', 'down', 'current'].index(t)] for t in doc_types]

In [6]:
# Нам, людям, проще визуализировать объекты в 2D!
# Уменьшите размерность векторов до 2D, используя t-SNE
# (t-распределенное стохастическое вложение соседей)

tsne = TSNE(n_components=2, random_state=42, perplexity=5)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vector Store Visualization',
    xaxis_title='x',
    yaxis_title='y',
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [7]:
# Let's try 3D!
tsne = TSNE(n_components=3, random_state=42, perplexity=5)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()
